# PA1 — Inferência

Recebe o caminho de **uma imagem qualquer** e devolve a **máscara de instâncias
colorida** e a **contagem de objetos**.

Não treina nada: carrega um checkpoint já treinado. Qualquer tamanho de imagem,
em RGB, RGBA ou tons de cinza.

## 1. Ambiente

No Colab, descomente o bloco do clone. Localmente, basta rodar a partir de `PA1/`.

In [ ]:
# --- Colab ---
# !git clone https://github.com/sofiaazeredo/deep-learning-2026.2.git
# %cd deep-learning-2026.2
# !git checkout structure
# %cd PA1

import sys
from pathlib import Path

if Path("src").exists() and "." not in sys.path:
    sys.path.insert(0, ".")

import numpy as np
import matplotlib.pyplot as plt

from src.inference import segment_image, default_checkpoint

print("checkpoint padrão:", default_checkpoint())

## 2. Escolha a imagem

Troque `IMAGE_PATH` por qualquer arquivo. O exemplo aponta para uma imagem do
conjunto de teste, mas pode ser uma imagem de fora do dataset.

`CHECKPOINT = None` usa o modelo final escolhido pela ablação de resolução
(`experiments/results/best_architecture.json`); passe um caminho para forçar outro.

In [ ]:
IMAGE_PATH = sorted(Path("data/raw").iterdir())[0]
IMAGE_PATH = next((IMAGE_PATH / "images").iterdir())

CHECKPOINT = None        # None = o melhor modelo da ablação
RESCUE_MARKERS = False   # True aplica a correção da Parte 5 (ver discussão lá)

print("imagem:", IMAGE_PATH)

## 3. Inferência

In [ ]:
result = segment_image(
    IMAGE_PATH,
    checkpoint=CHECKPOINT,
    rescue_markers=RESCUE_MARKERS,
)

print(f"checkpoint:   {result['checkpoint']}")
print(f"arquitetura:  {result['architecture']} ({result['out_channels']} canais)")
print(f"imagem:       {result['image'].shape[1]}x{result['image'].shape[0]} px")
print()
print(f"OBJETOS DETECTADOS: {result['count']}")

## 4. Máscara de instâncias colorida

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 6))

axes[0].imshow(result["image"])
axes[0].set_title("imagem de entrada")

axes[1].imshow(result["colored"])
axes[1].set_title(f"máscara de instâncias ({result['count']} objetos)")

axes[2].imshow(result["overlay"])
axes[2].set_title("sobreposição")

for ax in axes:
    ax.axis("off")

fig.tight_layout()
plt.show()

## 5. Salvar o resultado

A máscara é salva como PNG de 16 bits para não estourar em imagens com mais de
255 núcleos — o DSB2018 chega a 375 numa única imagem.

In [ ]:
from PIL import Image

OUTPUT_DIR = Path("experiments/figures/inferencia")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

stem = Path(IMAGE_PATH).stem

Image.fromarray(
    (result["colored"] * 255).astype(np.uint8)
).save(OUTPUT_DIR / f"{stem}_instancias_coloridas.png")

Image.fromarray(
    result["labels"].astype(np.uint16)
).save(OUTPUT_DIR / f"{stem}_instancias_ids.png")

print("salvo em:", OUTPUT_DIR)
print("contagem:", result["count"])

## 6. Lote (opcional)

Roda a mesma inferência sobre uma pasta inteira e imprime a contagem por imagem.

In [ ]:
def segment_folder(folder, pattern="*.png", checkpoint=None, limit=None):
    paths = sorted(Path(folder).glob(pattern))[:limit]

    for path in paths:
        out = segment_image(path, checkpoint=checkpoint)
        print(f"{path.name:<45} {out['count']:>4} objetos")


# segment_folder("data/raw/<algum_id>/images")